In [1]:
!pip3 install statsmodels


[notice] A new release of pip is available: 24.2 -> 24.3.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [2]:
import pandas as pd
import os

# Define file paths and corresponding model names
file_model_map = {
   # "detailed_metrics_test_model_final_20241219_004629.csv": "ResNetV2",
    "detailed_metrics_test_model_final_20250102_193709.csv": "ResNet50",
    "detailed_metrics_test_model_final_20241230_192828.csv": "MobileNetV2",
   # "detailed_metrics_test_model_final_20241226_023421.csv": "Xception",
    "detailed_metrics_test_model_final_20250101_221336.csv": "DenseNet121",
    "detailed_metrics_test_model_final_20250102_081842.csv": "EfficientNetV2B0",
    "detailed_metrics_test_model_final_20250102_090243.csv": "VGG16",
}

# Directory containing the files
directory = '/workspace/tables/'

# Initialize an empty list to store DataFrames
dataframes = []

# Loop through files and process each
for file, model in file_model_map.items():
    file_path = os.path.join(directory, file)
    
    # Read the CSV file
    df = pd.read_csv(file_path)
    
    # Add a column for the model name
    df['Model'] = model
    
    # Append to the list of DataFrames
    dataframes.append(df)

# Combine all DataFrames into a single DataFrame
combined_df_bosque = pd.concat(dataframes, ignore_index=True)
combined_df_bosque


,Metric,Value,Light,Dark,Model
0,Precision (Benign),0.767857,0.689655,0.851852,ResNet50
1,Precision (Malignant),0.779817,0.897436,0.483871,ResNet50
2,F1-Score (Benign),0.699187,0.701754,0.696970,ResNet50
3,F1-Score (Malignant),0.821256,0.891720,0.600000,ResNet50
4,Sensitivity,0.867347,0.886076,0.789474,ResNet50
5,Specificity,0.641791,0.714286,0.589744,ResNet50
6,Accuracy,0.775758,0.841121,0.655172,ResNet50
7,MCC,0.528054,0.593689,0.356808,ResNet50
8,AUC-ROC,0.846634,0.877034,0.775978,ResNet50
9,AUC-PR,0.901005,0.949392,0.749389,ResNet50


In [3]:

file_report_ham = {f"HAM10000_{key}": value for key, value in file_model_map.items()}

print(file_report_ham)

# Initialize an empty list to store DataFrames
dataframes_ham = []

# Loop through files and process each
for file, model in file_report_ham.items():
    file_path = os.path.join(directory, file)
    
    # Read the CSV file
    df = pd.read_csv(file_path)
    
    # Add a column for the model name
    df['Model'] = model
    
    # Append to the list of DataFrames
    dataframes_ham.append(df)

# Combine all DataFrames into a single DataFrame
combined_df_ham = pd.concat(dataframes_ham, ignore_index=True)
combined_df_ham.rename(columns={"Value": "HAM10000"}, inplace=True)
combined_df = pd.merge(combined_df_bosque, combined_df_ham)  

combined_df.rename(columns={"Value": "Overall"}, inplace=True)
combined_df = combined_df[["Model", "Metric","HAM10000", "Overall","Light","Dark"]]
combined_df

{'HAM10000_detailed_metrics_test_model_final_20250102_193709.csv': 'ResNet50', 'HAM10000_detailed_metrics_test_model_final_20241230_192828.csv': 'MobileNetV2', 'HAM10000_detailed_metrics_test_model_final_20250101_221336.csv': 'DenseNet121', 'HAM10000_detailed_metrics_test_model_final_20250102_081842.csv': 'EfficientNetV2B0', 'HAM10000_detailed_metrics_test_model_final_20250102_090243.csv': 'VGG16'}


,Model,Metric,HAM10000,Overall,Light,Dark
0,ResNet50,Precision (Benign),0.990842,0.767857,0.689655,0.851852
1,ResNet50,Precision (Malignant),0.750703,0.779817,0.897436,0.483871
2,ResNet50,F1-Score (Benign),0.799704,0.699187,0.701754,0.696970
3,ResNet50,F1-Score (Malignant),0.855312,0.821256,0.891720,0.600000
4,ResNet50,Sensitivity,0.993797,0.867347,0.886076,0.789474
5,ResNet50,Specificity,0.670384,0.641791,0.714286,0.589744
6,ResNet50,Accuracy,0.831990,0.775758,0.841121,0.655172
7,ResNet50,MCC,0.701798,0.528054,0.593689,0.356808
8,ResNet50,AUC-ROC,0.945351,0.846634,0.877034,0.775978
9,ResNet50,AUC-PR,0.929454,0.901005,0.949392,0.749389


In [4]:
# Reorder columns to place 'Model' first
columns = ['Model'] + [col for col in combined_df.columns if col != 'Model']
combined_df = combined_df[columns]

combined_df['PR (Light)'] = combined_df['Light'] - combined_df['Overall']
combined_df['PR (Dark)'] = combined_df['Dark'] - combined_df['Overall']

# Define the desired order of metrics based on importance
metric_order = [
    "Precision (Malignant)",
    "Sensitivity",
    "AUC-PR",
    "Specificity",
    "Accuracy",
    "AUC-ROC",
    "F1-Score (Malignant)",
    "MCC",
    "Precision (Benign)",
    "F1-Score (Benign)"
]

# Create a sorted order for the Model column based on Precision (Malignant)
precision_df = combined_df[combined_df['Metric'] == "Precision (Malignant)"]
precision_df = precision_df.sort_values(by="Overall", ascending=False)
sorted_model_order = precision_df['Model'].tolist()

# Create a categorical type to enforce the order for Metric and Model
combined_df['Metric'] = pd.Categorical(combined_df['Metric'], categories=metric_order, ordered=True)
combined_df['Model'] = pd.Categorical(combined_df['Model'], categories=sorted_model_order, ordered=True)

# Sort the DataFrame by the custom order
combined_df = combined_df.sort_values(by=['Metric', 'Model']).reset_index(drop=True)

# Display the sorted DataFrame
combined_df


,Model,Metric,HAM10000,Overall,Light,Dark,PR (Light),PR (Dark)
0,ResNet50,Precision (Malignant),0.750703,0.779817,0.897436,0.483871,0.117619,-0.295946
1,DenseNet121,Precision (Malignant),0.734448,0.691729,0.815217,0.414634,0.123488,-0.277095
2,MobileNetV2,Precision (Malignant),0.722394,0.684615,0.835294,0.400000,0.150679,-0.284615
3,EfficientNetV2B0,Precision (Malignant),0.677951,0.648000,0.797619,0.341463,0.149619,-0.306537
4,VGG16,Precision (Malignant),0.686620,0.597484,0.754902,0.315789,0.157418,-0.281695
5,ResNet50,Sensitivity,0.993797,0.867347,0.886076,0.789474,0.018729,-0.077873
6,DenseNet121,Sensitivity,0.981390,0.938776,0.949367,0.894737,0.010592,-0.044039
7,MobileNetV2,Sensitivity,0.868486,0.908163,0.898734,0.947368,-0.009429,0.039205
8,EfficientNetV2B0,Sensitivity,0.919355,0.826531,0.848101,0.736842,0.021571,-0.089689
9,VGG16,Sensitivity,0.967742,0.969388,0.974684,0.947368,0.005296,-0.022019


In [5]:
from statsmodels.stats.proportion import proportions_ztest

# Function to perform a z-test for proportions
def compute_proportion_test(row):
    # Extract proportions and sample sizes for Light and Dark
    light_value = row['Light']
    dark_value = row['Dark']
    light_samples = 107  # Replace with the actual number of samples for Light
    dark_samples = 58    # Replace with the actual number of samples for Dark

    # Convert proportions into counts
    count_light = light_value * light_samples
    count_dark = dark_value * dark_samples

    # Perform z-test for proportions
    count = [count_light, count_dark]
    nobs = [light_samples, dark_samples]

    z_stat, p_value = proportions_ztest(count, nobs)

    return z_stat, p_value

# Apply the test row-wise and create new columns for results
combined_df[['Z-Statistic', 'P-Value']] = combined_df.apply(
    lambda row: compute_proportion_test(row), axis=1, result_type='expand'
)

# Function to assign significance levels based on P-value
def assign_significance(p_value):
    if p_value < 0.001:
        return "***"
    elif p_value < 0.01:
        return "**"
    elif p_value < 0.05:
        return "*"
    else:
        return ""

# Apply the function to create a new 'Significance' column
combined_df['Significance'] = combined_df['P-Value'].apply(assign_significance)


combined_df

# Save detailed metrics to CSV
tables_dir = '/workspace/tables'
detailed_metrics_path = os.path.join(tables_dir, f'detailed_metrics_full.csv')
combined_df.to_csv(detailed_metrics_path, index=False)
print(f"Detailed metrics saved to {detailed_metrics_path}")

Detailed metrics saved to /workspace/tables/detailed_metrics_full.csv


/usr/local/lib/python3.8/dist-packages/statsmodels/stats/proportion.py:1025: RuntimeWarning: invalid value encountered in sqrt
  std_diff = np.sqrt(var_)


# Latex format

In [6]:
# Metrics you want to exclude
excluded_metrics = ['F1-Score (Benign)', 'MCC', 'Precision (Benign)']
combined_df = combined_df[~combined_df['Metric'].isin(excluded_metrics)]

In [7]:
import pandas as pd

# Step 1: Format columns to exactly 3 decimals or scientific notation for P-Value
columns_to_format = ['HAM10000','Overall', 'Light', 'Dark', 'PR (Light)', 'PR (Dark)', 'Z-Statistic']

# Convert to numeric (forcing errors to NaN) before formatting
for col in columns_to_format:
    combined_df[col] = pd.to_numeric(combined_df[col], errors='coerce')  # Convert to numeric, NaN for errors
    combined_df[col] = combined_df[col].map(lambda x: "{:.3f}".format(x) if pd.notnull(x) else "")  # Format or leave empty

# Format P-Value with scientific notation for small values
combined_df['P-Value'] = pd.to_numeric(combined_df['P-Value'], errors='coerce')  # Ensure numeric
combined_df['P-Value'] = combined_df['P-Value'].apply(
    lambda x: f"{x:.2e}" if pd.notnull(x) and x < 0.001 else (f"{x:.3f}" if pd.notnull(x) else "")
)

# Step 2: Remove rows with NaN in the 'Metric' column
cleaned_df = combined_df.dropna(subset=['Metric'])

# Step 3: Prepare a function to add metric rows and ensure correct column spans
def add_metric_rows_no_column(df):
    rows = []
    for metric, group in df.groupby('Metric'):
        # Add a row for the metric spanning all columns
        metric_row = [f"\\multicolumn{{9}}{{|c|}}{{\\textbf{{{metric}}}}} \\\\ \\hline"]
        rows.append(metric_row)
        # Add all rows of the group without the Metric column
        rows.extend(group.drop(columns=['Metric']).values.tolist())
    # Create a new DataFrame with updated structure
    columns = list(df.columns[1:])  # Exclude 'Metric' from column headers
    return pd.DataFrame(rows, columns=columns)

# Step 4: Apply the function to modify the DataFrame
modified_df = add_metric_rows_no_column(cleaned_df)

# Step 5: Generate LaTeX code from the modified DataFrame
latex_table = modified_df.to_latex(
    index=False,  # Do not include the index
    escape=False,  # Allow LaTeX formatting
    column_format="|l|c|c|c|c|c|c|c|c|c|",  # Adjust alignment for 10 columns
)

# Step 6: Replace table formatting for journal standards
latex_table = (
    "\\renewcommand{\\arraystretch}{1.2} % Adjust row spacing for readability\n"
    "\\tabcolsep=0.10cm\n"
    "\\begin{table}\n"
    "\\centering\n"
    "\\footnotesize\n"
    "\\caption{Performance metrics for various models evaluated on Light and Dark subsets. Significance levels are indicated: * P < 0.05, ** P < 0.01, *** P < 0.001.}\n"
    "\\label{tab:metrics_performance}\n"
    + latex_table.replace("\\toprule", "\\hline")
                 .replace("\\midrule", "\\hline")
                 .replace("\\bottomrule", "\\hline")
                 .replace("\\\\\n\\multicolumn", "\\\\ \\hline\n\\multicolumn")  # Ensure metric rows are separated by \hline
    + "\\end{table}"
)

# Step 7: Remove any rows with blank or malformed data
latex_table = latex_table.replace("NaN", "").strip()

# Step 8: Save and display the LaTeX table
with open("final_grouped_table.tex", "w") as f:
    f.write(latex_table)

print(latex_table)


/tmp/ipykernel_5481/2207897678.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  combined_df[col] = pd.to_numeric(combined_df[col], errors='coerce')  # Convert to numeric, NaN for errors
/tmp/ipykernel_5481/2207897678.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  combined_df[col] = combined_df[col].map(lambda x: "{:.3f}".format(x) if pd.notnull(x) else "")  # Format or leave empty
/tmp/ipykernel_5481/2207897678.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a D

\renewcommand{\arraystretch}{1.2} % Adjust row spacing for readability
\tabcolsep=0.10cm
\begin{table}
\centering
\footnotesize
\caption{Performance metrics for various models evaluated on Light and Dark subsets. Significance levels are indicated: * P < 0.05, ** P < 0.01, *** P < 0.001.}
\label{tab:metrics_performance}
\begin{tabular}{|l|c|c|c|c|c|c|c|c|c|}
\hline
Metric & HAM10000 & Overall & Light & Dark & PR (Light) & PR (Dark) & Z-Statistic & P-Value & Significance \\
\hline
\multicolumn{9}{|c|}{\textbf{Precision (Malignant)}} \\ \hline &  &  &  &  &  &  &  &  &  \\
ResNet50 & 0.751 & 0.780 & 0.897 & 0.484 & 0.118 & -0.296 & 5.874 & 4.26e-09 & *** \\
DenseNet121 & 0.734 & 0.692 & 0.815 & 0.415 & 0.123 & -0.277 & 5.243 & 1.58e-07 & *** \\
MobileNetV2 & 0.722 & 0.685 & 0.835 & 0.400 & 0.151 & -0.285 & 5.734 & 9.82e-09 & *** \\
EfficientNetV2B0 & 0.678 & 0.648 & 0.798 & 0.341 & 0.150 & -0.307 & 5.819 & 5.93e-09 & *** \\
VGG16 & 0.687 & 0.597 & 0.755 & 0.316 & 0.157 & -0.282 & 5.498 & 

In [8]:
modified_df

,Metric,HAM10000,Overall,Light,Dark,PR (Light),PR (Dark),Z-Statistic,P-Value,Significance
0,\multicolumn{9}{|c|}{\textbf{Precision (Malign...,None,None,None,None,None,None,None,None,None
1,ResNet50,0.751,0.780,0.897,0.484,0.118,-0.296,5.874,4.26e-09,***
2,DenseNet121,0.734,0.692,0.815,0.415,0.123,-0.277,5.243,1.58e-07,***
3,MobileNetV2,0.722,0.685,0.835,0.400,0.151,-0.285,5.734,9.82e-09,***
4,EfficientNetV2B0,0.678,0.648,0.798,0.341,0.150,-0.307,5.819,5.93e-09,***
5,VGG16,0.687,0.597,0.755,0.316,0.157,-0.282,5.498,3.83e-08,***
6,\multicolumn{9}{|c|}{\textbf{Sensitivity}} \\ ...,None,None,None,None,None,None,None,None,None
7,ResNet50,0.994,0.867,0.886,0.789,0.019,-0.078,1.669,0.095,
8,DenseNet121,0.981,0.939,0.949,0.895,0.011,-0.044,1.315,0.189,
9,MobileNetV2,0.868,0.908,0.899,0.947,-0.009,0.039,-1.074,0.283,
